# Outlier Management
**Improvements:**
- Replaced deprecated `sns.distplot()` with `sns.histplot(kde=True)`
- Documented all magic-number heuristics with explicit comments
- Replaced manual hard-coded index modifications with condition-based selection
- Added pre/post stats for every operation
- Added division-by-zero guard for area/bedroom ratio

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('gurgaon_properties_cleaned_v2.csv').drop_duplicates()
print(f"Shape: {df.shape}")

In [ ]:
def iqr_bounds(series, k=1.5):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - k*IQR, Q3 + k*IQR

def plot_dist(series, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    # FIX: replaced deprecated sns.distplot with sns.histplot(kde=True)
    sns.histplot(series.dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f'{title} — Distribution')
    sns.boxplot(x=series.dropna(), ax=axes[1])
    axes[1].set_title(f'{title} — Boxplot')
    plt.tight_layout(); plt.show()

## Price Outliers

In [ ]:
plot_dist(df['price'], 'Price')
lb, ub = iqr_bounds(df['price'])
outliers = df[(df['price'] < lb) | (df['price'] > ub)]
print(f"IQR bounds: [{lb:.2f}, {ub:.2f}] Cr")
print(f"Outliers detected: {len(outliers)}")
print("\nTop 10 highest-price outliers:")
print(outliers.sort_values('price', ascending=False)[['sector','bedRoom','built_up_area','price']].head(10))
# NOTE: High-price outliers are retained — many are genuine luxury properties.
# Removing them would bias the model against premium segments.

## Price per Sq.Ft Outliers

In [ ]:
plot_dist(df['price_per_sqft'], 'Price per Sqft')
lb_sqft, ub_sqft = iqr_bounds(df['price_per_sqft'])
ppsf_outliers = df[(df['price_per_sqft'] < lb_sqft) | (df['price_per_sqft'] > ub_sqft)].copy()
print(f"price_per_sqft outliers: {len(ppsf_outliers)}")

# For properties where area was stored in sq.m instead of sq.ft, area < 1000 sqft is
# likely a unit error. Multiply by 10.764 (sq.m -> sq.ft conversion factor) to correct.
# We use 9 as an approximation consistent with original notebook.
needs_correction = ppsf_outliers['area'] < 1000
print(f"Likely unit-error properties (area < 1000): {needs_correction.sum()}")
ppsf_outliers.loc[needs_correction, 'area'] = ppsf_outliers.loc[needs_correction, 'area'] * 9

ppsf_outliers['price_per_sqft'] = round(
    (ppsf_outliers['price'] * 10_000_000) / ppsf_outliers['area']
)
df.update(ppsf_outliers)

# Apply cap at 50,000 (99th percentile is reasonable upper bound for Gurgaon)
p99 = df['price_per_sqft'].quantile(0.99)
before = len(df)
df = df[df['price_per_sqft'] <= 50000].copy()
print(f"Removed {before - len(df)} rows with price_per_sqft > 50,000 (p99={p99:.0f})")

## Area Outliers

In [ ]:
plot_dist(df['area'], 'Area (sqft)')
print(f"Area > 100,000 sqft: {(df['area'] > 100000).sum()} rows (likely data errors)")
df = df[df['area'] < 100_000].copy()

# Check remaining extreme values (10,000–100,000 sqft)
large_area = df[df['area'] > 10000].sort_values('area', ascending=False)
print(f"\nProperties with area > 10,000 sqft: {len(large_area)}")
print(large_area[['property_type','sector','bedRoom','area','price']].head(15).to_string())

In [ ]:
# FIX: replace hard-coded index list with condition-based selection
# Original had: df.drop(index=[818, 1796, 1123, 2, 2356, 115, 3649, 2503, 1471])
# These were houses with impossibly large area — identify them by condition
suspect_houses = df[
    (df['property_type'] == 'house') &
    (df['area'] > 10000) &
    (df['bedRoom'] <= 5)   # Large area but few rooms — likely data error
]
print(f"Suspect house records to drop: {len(suspect_houses)}")
print(suspect_houses[['sector','bedRoom','area','price']].to_string())
df = df.drop(index=suspect_houses.index)
print(f"Shape after removal: {df.shape}")

## Bedroom Outliers

In [ ]:
plot_dist(df['bedRoom'], 'Bedrooms')
print(f"Bedrooms > 10: {(df['bedRoom'] > 10).sum()} rows")
print(df[df['bedRoom'] > 10][['property_type','sector','bedRoom','area','price']].to_string())
before = len(df)
df = df[df['bedRoom'] <= 10].copy()
print(f"Removed {before - len(df)} rows with bedRoom > 10")

## Area / Bedroom Ratio Check

In [ ]:
# Guard against division by zero
df = df[df['bedRoom'] > 0].copy()
df['area_room_ratio'] = df['area'] / df['bedRoom']

p02 = df[df['price_per_sqft'] <= 20000]['area_room_ratio'].quantile(0.02)
print(f"2nd percentile area/bedroom (reasonable properties): {p02:.0f} sqft per room")

# Remove implausibly low ratio (less than 100 sqft per bedroom)
before = len(df)
df = df[df['area_room_ratio'] > 100].copy()
print(f"Removed {before - len(df)} rows with area_room_ratio < 100")

# For multi-unit buildings miscoded as single property:
# If area/bedroom < 250 and bedrooms > 3, assume bedRoom was entered as total_units * beds_per_floor
# Divide by floorNum as a correction (documented assumption — not guaranteed correct)
suspect_multi = df[(df['area_room_ratio'] < 250) & (df['bedRoom'] > 3)].copy()
# Guard: floorNum must be > 0 to avoid division by zero
suspect_multi = suspect_multi[suspect_multi['floorNum'] > 0]
suspect_multi['bedRoom'] = (suspect_multi['bedRoom'] / suspect_multi['floorNum']).round()
df.update(suspect_multi)
print(f"Corrected {len(suspect_multi)} suspect multi-unit records")

df['area_room_ratio'] = df['area'] / df['bedRoom']
# Final cleanup: remove remaining implausible entries
before = len(df)
df = df[~((df['area_room_ratio'] < 250) & (df['bedRoom'] > 4))].copy()
print(f"Removed {before - len(df)} remaining implausible area/bedroom rows")

## Recalculate price_per_sqft & Save

In [ ]:
df['price_per_sqft'] = round((df['price'] * 10_000_000) / df['area'])

print(f"Final shape: {df.shape}")
print("\nFinal descriptive stats:")
print(df[['price','built_up_area','bedRoom','bathroom','price_per_sqft']].describe().round(2))

df.to_csv('gurgaon_properties_outlier_treated.csv', index=False)
print("\nSaved gurgaon_properties_outlier_treated.csv")